# AI Predictive Maintenance & Machine Health Analytics System

## 1. Environment Configuration and Database Connectivity

### Overview
This section initializes the analytical environment and establishes a secure connection between Python and the SQL Server instance hosting the industrial sensor dataset.

The objective is to create a seamless data pipeline enabling structured data retrieval, preprocessing, modeling, and business reporting.

## Library Imports

The following libraries are imported to support different stages of the pipeline:

- **pandas**: Data manipulation and tabular analysis  
- **numpy**: Numerical operations and vectorized computations  
- **pyodbc**: ODBC-based connectivity to SQL Server  
- **matplotlib / seaborn**: Statistical visualization and exploratory analysis  

These libraries collectively enable end-to-end implementation of the predictive maintenance workflow.


In [1]:
import pandas as pd
import numpy as np
import pyodbc   
import matplotlib.pyplot as plt
import seaborn as sns


### Database Connection (pyodbc)

pyodbc is a Python library that enables database connectivity through the ODBC (Open Database Connectivity) interface.

In this project, it is used to establish a secure connection between Python and the SQL Server instance hosting the industrial dataset.

This connection allows:
- Executing SQL queries  
- Retrieving structured data into Python  
- Integrating SQL with the ML pipeline  

Successful execution confirms that the system is ready for database-driven data operations.

pyodbc.connect() -> creates a bridge between Python and SQL Server
SERVER= -> your SQL instance
DATABASE= -> which database to use
Trusted_Connection=yes -> use Windows login

In [2]:
conn = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=MAYURESH4U\\SQLEXPRESS;'
    'DATABASE=PredictiveMaintenanceDB;'
    'Trusted_Connection=yes;'
)

print("Connected to SQL Server successfully")


Connected to SQL Server successfully


## 2. Data Extraction from SQL Server

### SQL Query Execution

This step retrieves the complete industrial machine dataset from the ML_MachineFailureData table using a structured SQL query.

```sql
SELECT * FROM ML_MachineFailureData

In [3]:
query = "SELECT * FROM ML_MachineFailureData"

df = pd.read_sql(query, conn)

df.head()


C:\Users\MAYURESH PARCHE\AppData\Local\Temp\ipykernel_18700\1030469489.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,Type,Air_temperature,Process_temperature,Rotational_speed,Torque,Tool_wear,Machine_failure
0,M,298.1,308.6,1551.0,42.8,0,0
1,L,298.2,308.7,1408.0,46.3,3,0
2,L,298.1,308.5,1498.0,49.4,5,0
3,L,298.2,308.6,1433.0,39.5,7,0
4,L,298.2,308.7,1408.0,40.0,9,0


In [4]:
df.shape


(10000, 7)

In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Type                 10000 non-null  object 
 1   Air_temperature      10000 non-null  float64
 2   Process_temperature  10000 non-null  float64
 3   Rotational_speed     10000 non-null  float64
 4   Torque               10000 non-null  float64
 5   Tool_wear            10000 non-null  int64  
 6   Machine_failure      10000 non-null  int64  
dtypes: float64(4), int64(2), object(1)
memory usage: 547.0+ KB


## 3. Categorical Feature Encoding

### One-Hot Encoding

The categorical feature (`Type`) is transformed into numerical format using one-hot encoding.

Since machine learning algorithms operate on numerical inputs, categorical levels are converted into binary indicator variables via:

```python
pd.get_dummies(df, columns=['Type'], drop_first=True)


In [6]:
df = pd.get_dummies(df, columns=['Type'], drop_first=True)

df.head()

,Air_temperature,Process_temperature,Rotational_speed,Torque,Tool_wear,Machine_failure,Type_L,Type_M
0,298.1,308.6,1551.0,42.8,0,0,False,True
1,298.2,308.7,1408.0,46.3,3,0,True,False
2,298.1,308.5,1498.0,49.4,5,0,True,False
3,298.2,308.6,1433.0,39.5,7,0,True,False
4,298.2,308.7,1408.0,40.0,9,0,True,False


## 4. Feature–Target Separation

### Defining Input Features (X) and Target Variable (y)

The dataset is partitioned into:

- **X (Features):** All predictor variables used for training  
- **y (Target):** Binary failure indicator (`Machine_failure`)  

The model will learn the relationship betweeen sensor data and machine failure

In [7]:
X = df.drop('Machine_failure', axis=1)  #axis=1 means droppinng a column and 0 means a row
y = df['Machine_failure']  #This is our Target variable 

# Check shapes
print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (10000, 7)
y shape: (10000,)


## 5. Train–Test Split

### Dataset Partitioning

The dataset is divided into training and testing subsets to evaluate model generalization performance.

- **80%** of the data is allocated for training  
- **20%** is reserved for testing  

```python
train_test_split(X, y, test_size=0.2, random_state=42)


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


X_train shape: (8000, 7)
X_test shape: (2000, 7)
y_train shape: (8000,)
y_test shape: (2000,)


## 6. Logistic Regression Model Initialization

Logistic Regression is imported for binary classification.  
It will be used to predict whether a machine failure occurs (1) or not (0).


In [9]:
from sklearn.linear_model import LogisticRegression


## 7. Model Training

A Logistic Regression model is initialized with max_iter=1000 to ensure convergence.

The model is trained using model.fit(X_train, y_train), allowing it to learn the relationship between sensor features and machine failure..


In [10]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


## 8. Model Prediction

The trained Logistic Regression model is used to generate predictions on the test dataset using model.predict(X_test).

The output consists of binary labels:
0 = No failure  
1 = Machine failure

In [11]:
y_pred = model.predict(X_test)

y_pred[:10]


array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

## 9. Model Performance Evaluation

Model performance is evaluated using accuracy_score, which measures the proportion of correct predictions on the test set.

Test Accuracy: 0.974

In [12]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
accuracy


0.974

## 10. Confusion Matrix

The confusion matrix is computed to evaluate classification performance in terms of:

- True Positives  
- True Negatives  
- False Positives  
- False Negatives  

This provides a detailed breakdown of prediction outcomes beyond overall accuracy.

In [13]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
cm


array([[1930,    9],
       [  43,   18]])

## 11. Classification Report

The classification report summarizes precision, recall, and F1-score for each class.

This provides a more comprehensive evaluation, particularly for imbalanced datasets, where accuracy alone may be misleading.

In [14]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1939
           1       0.67      0.30      0.41        61

    accuracy                           0.97      2000
   macro avg       0.82      0.65      0.70      2000
weighted avg       0.97      0.97      0.97      2000



## 12. Class Imbalance Handling (Balanced Logistic Regression)

A new Logistic Regression model is trained using class_weight='balanced' to address class imbalance.

This adjusts weights inversely proportional to class frequencies, improving recall for the minority class.

In [15]:
model_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
model_balanced.fit(X_train, y_train)

y_pred_bal = model_balanced.predict(X_test)

print(classification_report(y_test, y_pred_bal))


              precision    recall  f1-score   support

           0       0.99      0.82      0.90      1939
           1       0.13      0.85      0.22        61

    accuracy                           0.82      2000
   macro avg       0.56      0.83      0.56      2000
weighted avg       0.97      0.82      0.88      2000



## 13. ROC-AUC Evaluation

Predicted probabilities are computed using predict_proba to evaluate model discrimination ability.

The ROC-AUC score measures how well the model distinguishes between failure and non-failure cases.

ROC-AUC Score: 0.9013

In [16]:
from sklearn.metrics import roc_auc_score

y_prob = model_balanced.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)
roc_auc


0.9012842516423034

## 14. Feature Importance Interpretation

Model coefficients from the balanced Logistic Regression are analyzed to assess feature influence on failure prediction.

Positive coefficients indicate increased failure probability, while negative coefficients indicate an inverse relationship.

Air_temperature shows the strongest positive association with failure risk, whereas Process_temperature exhibits a negative relationship.

This interpretation supports sensor-level insight for predictive maintenance decision-making.

In [17]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model_balanced.coef_[0]
})

feature_importance.sort_values(by="Coefficient", ascending=False)


,Feature,Coefficient
0,Air_temperature,0.902068
5,Type_L,0.623723
6,Type_M,0.364658
3,Torque,0.247189
4,Tool_wear,0.014855
2,Rotational_speed,0.009887
1,Process_temperature,-0.813332


## 15. Threshold Tuning for Business-Oriented Optimization

The default classification threshold of 0.5 is adjusted to 0.3 to prioritize failure detection.

In predictive maintenance, missing a failure (false negative) is more costly than triggering a false alarm. Lowering the threshold increases recall for the failure class at the expense of precision.

This step demonstrates business-driven model calibration rather than relying solely on default decision boundaries.

In [18]:
import numpy as np

# Use predicted probabilities
y_prob = model_balanced.predict_proba(X_test)[:, 1]

# Custom threshold
threshold = 0.3
y_pred_custom = (y_prob >= threshold).astype(int)

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_custom))


              precision    recall  f1-score   support

           0       0.99      0.69      0.81      1939
           1       0.08      0.89      0.15        61

    accuracy                           0.69      2000
   macro avg       0.54      0.79      0.48      2000
weighted avg       0.97      0.69      0.79      2000



The trained model is saved using joblib so it can be deployed in production without retraining.

In [19]:
import joblib

joblib.dump(model_balanced, "machine_failure_model.pkl")


['machine_failure_model.pkl']

## 15. Raw Dataset Loading

The original dataset is loaded from the CSV file using pandas.

This step allows validation of the raw data structure before any preprocessing or database integration.

df_original.head() is used to preview the first few records and confirm column structure, feature names, and target variables.

In [25]:
import pandas as pd
import numpy as np

df_original = pd.read_csv("ai4i2020.csv")
df_original.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 16. Full Dataset Preparation for Deployment

The original dataset is re-encoded to ensure consistency with the training pipeline.

One-hot encoding is applied to the categorical feature, and the target variable is removed to construct X_full exactly as used during model training.

This guarantees that the feature structure during inference matches the structure used during model development.

In [29]:
# Encode Type again
df_encoded = pd.get_dummies(df_original, columns=["Type"], drop_first=True)

# Create X_full exactly like training
X_full = df_encoded.drop("Machine failure", axis=1)

X_full.head()

,UDI,Product ID,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],TWF,HDF,PWF,OSF,RNF,Type_L,Type_M
0,1,M14860,298.1,308.6,1551,42.8,0,0,0,0,0,0,False,True
1,2,L47181,298.2,308.7,1408,46.3,3,0,0,0,0,0,True,False
2,3,L47182,298.1,308.5,1498,49.4,5,0,0,0,0,0,True,False
3,4,L47183,298.2,308.6,1433,39.5,7,0,0,0,0,0,True,False
4,5,L47184,298.2,308.7,1408,40.0,9,0,0,0,0,0,True,False


## 17. Deployment Inference Pipeline

Column names are standardized to match the training schema, and categorical encoding is reapplied to ensure feature consistency.

The feature set is aligned using model.feature_names_in_ to guarantee identical structure and ordering as during training.

Failure probabilities are then generated for the complete dataset using the trained model.

In [36]:
df_original = df_original.rename(columns={
    "Air temperature [K]": "Air_temperature",
    "Process temperature [K]": "Process_temperature",
    "Rotational speed [rpm]": "Rotational_speed",
    "Torque [Nm]": "Torque",
    "Tool wear [min]": "Tool_wear"
})

In [37]:
df_encoded = pd.get_dummies(df_original, columns=["Type"], drop_first=True)

In [38]:
X_full = df_encoded[model_balanced.feature_names_in_]

In [39]:
y_prob_full = model_balanced.predict_proba(X_full)[:, 1]

## 18. Risk Scoring and Export

Predicted probabilities are compiled into a structured output containing unique identifiers and failure risk scores.

A High_Risk flag is generated using the optimized threshold (0.3), and the final predictions are exported for operational use.

In [40]:
prediction_df = pd.DataFrame({
    "UDI": df_original["UDI"],
    "Failure_Probability": y_prob_full
})

# Create High Risk flag
prediction_df["High_Risk"] = np.where(
    prediction_df["Failure_Probability"] >= 0.3, 1, 0
)

prediction_df.head()

,UDI,Failure_Probability,High_Risk
0,1,0.046742,0
1,2,0.037275,0
2,3,0.183452,0
3,4,0.010515,0
4,5,0.008840,0


In [41]:
prediction_df.to_csv("machine_predictions.csv", index=False)